In [10]:
%load_ext autoreload
%autoreload 2

In [11]:
import torch.nn as nn
import torch.nn.functional as F
import torch

# TODO:

- Multiheaded attention
- Autoregressive decoder

In [12]:
torch.cuda.is_available()

True

# Data Loading

In [13]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

# Load the Multi30k dataset (English-German)
dataset = load_dataset("opus_books", 'de-en')

# Load a pretrained tokenizer (e.g., MarianMT or T5 works great for translation)
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

/home/ahmad/miniforge3/envs/machine-learning/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [14]:
def tokenize_function(examples):
    # Get list of English and German sentences
    src_texts = [example["en"] for example in examples["translation"]]
    tgt_texts = [example["de"] for example in examples["translation"]]

    # Tokenize input (English)
    model_inputs = tokenizer(src_texts, max_length=128, truncation=True, padding="max_length")

    # Tokenize target (German)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(tgt_texts, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["translation"])

In [15]:
print(dataset["train"]["translation"][10])
print(dataset["train"]["id"][10])

{'de': '»Jane, ich liebe weder Spitzfindigkeiten noch Fragen; außerdem ist es gradezu widerlich, wenn ein Kind ältere Leute in dieser Weise zur Rede stellt.', 'en': '"Jane, I don\'t like cavillers or questioners; besides, there is something truly forbidding in a child taking up her elders in that manner.'}
10


In [16]:
vocab = tokenizer.get_vocab()
first_20_tokens = list(vocab.items())[:20]

# Display first 20 token-vocab pairs
for token, idx in first_20_tokens:
    print(f"Token: {token}, Index: {idx}")

sentence = '"Jane, I don\'t like cavillers or questioners; besides, there is something truly forbidding in a child taking up her elders in that manner.'

# Encode the sentence to token IDs
encoded = tokenizer.encode(sentence)

print(f"Encoded sentence: {encoded}")

Token: </s>, Index: 0
Token: <unk>, Index: 1
Token: ,, Index: 2
Token: ., Index: 3
Token: ▁the, Index: 4
Token: ▁in, Index: 5
Token: s, Index: 6
Token: ▁of, Index: 7
Token: ▁and, Index: 8
Token: ▁der, Index: 9
Token: ▁und, Index: 10
Token: ▁die, Index: 11
Token: ▁to, Index: 12
Token: -, Index: 13
Token: ▁a, Index: 14
Token: en, Index: 15
Token: :, Index: 16
Token: ▁, Index: 17
Token: e, Index: 18
Token: ▁is, Index: 19
Encoded sentence: [47, 821, 4934, 2, 38, 413, 22, 46, 209, 1262, 4753, 2137, 62, 891, 462, 77, 17404, 2, 169, 19, 919, 7312, 38251, 3490, 5, 14, 2372, 1529, 150, 249, 27533, 5, 35, 4863, 3, 0]


In [17]:
# tokenized_dataset["train"]
# tokenized_dataset["train"]["labels"][10]

In [18]:
# Setup dataloader
from torch.utils.data import DataLoader

def collate_fn(batch):
    input_ids = torch.tensor([item['input_ids'] for item in batch])
    attention_mask = torch.tensor([item['attention_mask'] for item in batch])
    labels = torch.tensor([item['labels'] for item in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_loader = DataLoader(tokenized_dataset["train"], batch_size=32, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8)

In [19]:
import math

class SelfAttention(nn.Module):
  def __init__(self, hidden_dim, embedding_dim) -> None:
    super(SelfAttention, self).__init__()

    self.hidden_dim = hidden_dim
    self.embedding_dim = embedding_dim

    self.key = nn.Linear(embedding_dim, hidden_dim)
    self.query = nn.Linear(embedding_dim, hidden_dim)
    self.value = nn.Linear(embedding_dim, hidden_dim)


  def forward(self, x, attention_mask):
    x_key = self.key(x)
    x_query = self.query(x)

    scores = torch.bmm(x_query, x_key.transpose(1, 2))
    scaled_scores = scores / math.sqrt(self.hidden_dim)

    # Handle both mask shapes
    if attention_mask.dim() == 2:
        # (batch, seq_len) -> (batch, 1, seq_len)
        attn_mask = attention_mask.unsqueeze(1)
        scaled_scores = scaled_scores.masked_fill(attn_mask == 0, -1e9)
    elif attention_mask.dim() == 3:
        # (batch, seq_len, seq_len)
        scaled_scores = scaled_scores.masked_fill(attention_mask == 0, -1e9)
    weights = F.softmax(scaled_scores, dim=-1)

    """
    print('SCALED SCORES BEFORE: ', scaled_scores.shape)
    attn_mask = attention_mask.unsqueeze(1)
    scaled_scores = scaled_scores.masked_fill(attn_mask == 0, -1e9)
    print('SCALED SCORES AFTER: ', scaled_scores.shape)
    weights = F.softmax(scaled_scores, dim=1)
    """
    x_value = self.value(x)
    output = torch.bmm(weights, x_value)

    return output


In [20]:
class CrossAttention(nn.Module):
    def __init__(self, hidden_dim, embedding_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.key = nn.Linear(embedding_dim, hidden_dim)
        self.query = nn.Linear(embedding_dim, hidden_dim)
        self.value = nn.Linear(embedding_dim, hidden_dim)

    def forward(self, x, encoder_output, encoder_mask):
        # x: (B, L_tgt, D), encoder_output: (B, L_src, D)
        q = self.query(x)
        k = self.key(encoder_output)
        v = self.value(encoder_output)
        scores = torch.bmm(q, k.transpose(1, 2)) / (self.hidden_dim ** 0.5)
        if encoder_mask is not None:
            encoder_mask = encoder_mask.unsqueeze(1)
            scores = scores.masked_fill(encoder_mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        output = torch.bmm(weights, v)
        return output

In [21]:
class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_len, embedding_dim):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embedding_dim)

    def forward(self, x):
        # x: (B, L, D)
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand_as(x[:, :, 0])
        return x + self.pos_embedding(positions)

In [22]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, embedding_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-math.log(10000.0) / embedding_dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # shape (1, max_len, embedding_dim)
        self.register_buffer('pe', pe)  # not a parameter

    def forward(self, x):
        # x: (batch_size, seq_len, embedding_dim)
        x = x + self.pe[:, :x.size(1), :]
        return x

In [23]:
class EncoderBlock(nn.Module):
  def __init__(self, hidden_dim, embedding_dim, n_heads):
    super(EncoderBlock, self).__init__()

    self.self_attention_heads = nn.ModuleList(SelfAttention(hidden_dim=hidden_dim, embedding_dim=embedding_dim) for _ in range(n_heads))
    self.merge_layer = nn.Linear(n_heads*hidden_dim, hidden_dim)

    self.dim_feedforward = hidden_dim // 2
    self.n_heads = n_heads

    self.linear_net = nn.Sequential(
            nn.Linear(hidden_dim, self.dim_feedforward),
            nn.ReLU(inplace=True),
            nn.Linear(self.dim_feedforward, hidden_dim),
        )

    self.norm1 = nn.LayerNorm(hidden_dim)
    self.norm2 = nn.LayerNorm(hidden_dim)

  def forward(self, x, attention_mask):

    outputs = [self.self_attention_heads[i](x, attention_mask) for i in range(self.n_heads)]
    outputs = torch.cat(outputs, dim=-1)
    x_encoded = self.merge_layer(outputs)
    x_encoded = self.norm1(x_encoded + x)

    x_linear = self.linear_net(x_encoded)
    x_linear = self.norm2(x_linear + x_encoded)

    return x_linear

In [24]:
class DecoderBlock(nn.Module):
    def __init__(self, hidden_dim, embedding_dim, n_heads):
        super().__init__()
        self.self_attn = nn.ModuleList(SelfAttention(hidden_dim=hidden_dim, embedding_dim=embedding_dim) for _ in range(n_heads))
        self.merge_layer_self = nn.Linear(n_heads*hidden_dim, hidden_dim)
        self.cross_attn = nn.ModuleList(CrossAttention(hidden_dim=hidden_dim, embedding_dim=embedding_dim) for _ in range(n_heads))
        self.merge_layer_cross = nn.Linear(n_heads*hidden_dim, hidden_dim)

        self.linear_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)

        self.n_heads = n_heads

    def forward(self, x, encoder_output, tgt_mask, encoder_mask):
        # Masked self-attention
        x2 = [self.self_attn[i](x, tgt_mask) for i in range(self.n_heads)]
        x2 = torch.cat(x2, dim=-1)
        x2 = self.merge_layer_self(x2)
        x = self.norm1(x + x2)
        # Cross-attention
        x2 = [self.cross_attn[i](x, encoder_output, encoder_mask) for i in range(self.n_heads)]
        x2 = torch.cat(x2, dim=-1)
        x2 = self.merge_layer_cross(x2)
        x = self.norm2(x + x2)
        # Feedforward
        x2 = self.linear_net(x)
        x = self.norm3(x + x2)
        return x

In [25]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, n_layers, hidden_dim, embedding_dim, n_heads):
    super(Encoder, self).__init__()

    self.n_layers = n_layers

    self.encoder_blocks = []

    for _ in range(n_layers):
      self.encoder_blocks.append(EncoderBlock(hidden_dim=hidden_dim, embedding_dim=embedding_dim, n_heads=n_heads))

    self.encoder_blocks = nn.Sequential(*self.encoder_blocks)
    self.vocab_size = vocab_size
    self.embedding = nn.Embedding(self.vocab_size, embedding_dim)
    self.pos_encoding = PositionalEncoding(max_len=128, embedding_dim=embedding_dim)

  def forward(self, x, attention_mask):
    x_embedding = self.embedding(x)
    x_encoded = self.pos_encoding(x_embedding)

    for i in range(self.n_layers):
      x_encoded = self.encoder_blocks[i](x_encoded, attention_mask)

    return x_encoded

In [26]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, n_layers, hidden_dim, embedding_dim, max_len, n_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding = PositionalEncoding(embedding_dim, max_len=max_len)
        self.layers = nn.ModuleList([DecoderBlock(hidden_dim, embedding_dim, n_heads=n_heads) for _ in range(n_layers)])
        self.out_probs = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt, encoder_output, tgt_mask, encoder_mask):
        x = self.embedding(tgt)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, encoder_output, tgt_mask, encoder_mask)
        return self.out_probs(x)

In [27]:
class Transformer(nn.Module):
  def __init__(self, vocab_size, n_layers=12, hidden_dim=256, embedding_dim=256, n_heads=12, max_len=128):
    super(Transformer, self).__init__()

    self.encoder = Encoder(vocab_size, n_layers, hidden_dim=hidden_dim, embedding_dim=embedding_dim, n_heads=n_heads)
    self.decoder = Decoder(vocab_size, n_layers, hidden_dim=hidden_dim, embedding_dim=embedding_dim, max_len=max_len, n_heads=n_heads)

  def forward(self, source, target, source_mask, target_mask):
    x_encoded = self.encoder(source, source_mask)
    return self.decoder(target, x_encoded, target_mask, source_mask)
    

In [28]:
from torch.optim.lr_scheduler import LambdaLR
import pytorch_lightning as pl
from torch.optim import AdamW, Adam


class TranslationModel(pl.LightningModule):
    def __init__(self, train_loader, val_loader, test_loader, target_lr=1e-4, warmup_steps=4000):
        super().__init__()

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.warmup_steps = warmup_steps

        self.target_lr = target_lr
        self.vocab_size = tokenizer.vocab_size
        self.model = Transformer(vocab_size=self.vocab_size)
        self.criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    def train_dataloader(self):
        return self.train_loader

    def val_dataloader(self):
        return self.val_loader

    def test_dataloader(self):
        return self.test_loader

    @staticmethod   
    def generate_causal_mask(seq_len, batch_size, device):
        # Create a lower-triangular matrix (tgt_len, tgt_len)
        mask = torch.tril(torch.ones((seq_len, seq_len), device=device))
        # Expand to (batch_size, tgt_len, tgt_len)
        mask = mask.unsqueeze(0).expand(batch_size, -1, -1)
        return mask

    def forward(self, source, target, source_mask, target_mask):
        return self.model(source, target, source_mask, target_mask)

    
    def step(self, batch, batch_idx):
        # Extract input and target sequences from the batch
        input_ids, source_mask, labels = batch["input_ids"], batch["attention_mask"], batch["labels"]
        
        decoder_input_ids = labels[:, :-1]
        decoder_labels = labels[:, 1:]

        # Causal mask for decoder
        tgt_mask = self.generate_causal_mask(decoder_input_ids.size(1), input_ids.shape[0], decoder_input_ids.device)

        # Forward pass
        outputs = self.model(input_ids, decoder_input_ids, source_mask, tgt_mask)  # (B, tgt_len-1, vocab_size)
        loss = self.criterion(outputs.permute(0, 2, 1), decoder_labels)
        return loss

    def training_step(self, batch, batch_idx):
        loss = self.step(batch, batch_idx)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        self.log("lr", self.trainer.optimizers[0].param_groups[0]['lr'], prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self.step(batch, batch_idx)
        self.log("validation_loss", loss)
        return loss

    def configure_optimizers(self):
        optimizer = Adam(self.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

        # Calculate scaling factor to hit `target_lr` at the end of warmup
        scale_factor = self.target_lr * self.warmup_steps ** 0.5

        def lr_lambda(step: int):
            step = max(step, 1)
            return scale_factor * min(step ** -0.5, step * self.warmup_steps ** -1.5)

        scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
            }
        }

from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=30,
    accelerator="auto",  # use GPU if available
)

pl_model = TranslationModel(train_loader=train_loader, val_loader=train_loader, test_loader=train_loader)
trainer.fit(model=pl_model)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
2025-06-17 21:53:36.092191: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-17 21:53:36.100518: E external/local_xla/xla/stream_executor

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/ahmad/miniforge3/envs/machine-learning/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:476: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.


In [ ]:
# def predict(x, attn_mask, model):
#   probs = model.cuda()(x, attn_mask)
#   output = torch.argmax(probs, dim=-1)
#   print(output.detach().cpu().numpy())
#   return tokenizer.decode(output.detach().cpu().numpy())

# sample_input = 'she is Janne'
# sample_token = tokenizer(sample_input, max_length=128, truncation=True, padding="max_length")
# sample_encoded = torch.tensor(sample_token["input_ids"]).cuda()
# attention_mask = torch.tensor(sample_token["attention_mask"]).cuda()
# sample_encoded = sample_encoded.unsqueeze(0)
# attention_mask = attention_mask.unsqueeze(0)

# predict(sample_encoded, attention_mask, pl_model)

TypeError: TranslationModel.forward() missing 2 required positional arguments: 'source_mask' and 'target_mask'

In [44]:
def predict(sentence, model, tokenizer, max_len=128, device="cuda"):
    model.to(device)
    model.eval()
    # Tokenize source sentence
    tokenized = tokenizer(sentence, return_tensors="pt", max_length=max_len, truncation=True, padding="max_length")
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)

    # Prepare decoder input (start with BOS or pad token)
    if tokenizer.bos_token_id is not None:
        start_token = tokenizer.bos_token_id
    else:
        start_token = tokenizer.pad_token_id
    decoder_input = torch.empty(size=(0, 0), dtype=torch.long, device=device)

    # Autoregressive decoding loop
    for _ in range(max_len - 1):
        tgt_mask = torch.tril(torch.ones((1, decoder_input.size(1), decoder_input.size(1)), device=device))
        with torch.no_grad():
            output = model(input_ids, decoder_input, attention_mask, tgt_mask)
        next_token_logits = output[:, -1, :]  # (batch, vocab)
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)  # (batch, 1)
        decoder_input = torch.cat([decoder_input, next_token], dim=1)
        # Stop if EOS token is generated
        if tokenizer.eos_token_id is not None and next_token.item() == tokenizer.eos_token_id:
            break
    # Remove start token and decode
    output_tokens = decoder_input[0].tolist()
    if output_tokens[0] == start_token:
        output_tokens = output_tokens[1:]
    # Stop at EOS if present
    if tokenizer.eos_token_id in output_tokens:
        output_tokens = output_tokens[:output_tokens.index(tokenizer.eos_token_id)]
    return tokenizer.decode(output_tokens, skip_special_tokens=True)

# Example usage:
translated = predict("my name is", pl_model, tokenizer)
print(translated)

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [0, 256] but got: [1, 256].

In [ ]:
dir(pl_model)